# Chicago 2024: Datenqualität Schritt für Schritt

Dieses Notebook prüft die Bronze-Schicht des Jahrgangs 2024 — Schritt für
Schritt, jeder mit einer Frage, einer Prüfung und einem Befund.

**Wichtig für die Einordnung:** Bronze ist unverändert aus der API übernommen.
Es wird nichts geparst, nichts korrigiert, nichts weggeworfen. Was hier auffällt,
ist deshalb eine Eigenschaft **der Quelle**, nicht ein Fehler der Ingestion.
Genau das ist der Zweck der Schicht: Erst messen, dann bereinigen — das
Bereinigen gehört nach Silver.

Am Ende steht eine Übersicht aller Prüfungen mit Ampelstatus.

| | |
| --- | --- |
| Datenbestand | `data/bronze/city=chicago/`, alle Läufe des Jahrgangs 2024 |
| Grundlage | festes Schema aus `chicago_pipeline.download_chicago` |
| Referenz | `config/iucr_nibrs_mapping.csv` |

## Schritt 1 – Daten laden

Gelesen werden **alle** Läufe gemeinsam. Das ist möglich, weil das Schema
gepinnt ist: Voller Snapshot und Delta-Läufe teilen dieselben 23 Spalten.

`BEFUNDE` sammelt unterwegs das Ergebnis jeder Prüfung ein — als Dictionary,
damit ein erneut ausgeführte Zelle den Eintrag ersetzt statt ihn zu verdoppeln.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import json
import sys

import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'chicago_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'chicago_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/chicago_pipeline nicht gefunden')

sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))
from chicago_pipeline import download_chicago as dc

JAHR = 2024
DATA_ROOT = PROJECT_ROOT / 'data'
BEFUNDE: dict[str, tuple[str, str]] = {}

teile = sorted((DATA_ROOT / f'bronze/city={dc.CITY}').glob('**/' + dc.PARQUET_FILE))
if not teile:
    raise RuntimeError(
        'Keine Bronze-Läufe gefunden. Erst laden mit:\n'
        '  python -m chicago_pipeline.download_chicago --year 2024'
    )

tabelle = pq.read_table(teile, schema=dc.SCHEMA)
roh = tabelle.to_pylist()

print('Parquet-Dateien :', len(teile))
print('Zeilen gesamt   :', f'{len(roh):,}')
print('Spalten         :', tabelle.num_columns)

## Schritt 2 – Was das Manifest bereits beweist

Datenqualität beginnt **vor** der ersten Zeile: Wurde überhaupt vollständig
geladen? Diese Frage beantworten die Daten nicht, sondern das Manifest.

Die Ingestion schreibt die Menge der `id`-Werte fest, **bevor** sie lädt, und
gleicht danach ab. `completeness.verified` sagt, ob diese Kette durchlaufen
wurde; `source_changed_during_run` dokumentiert, ob sich die Quelle währenddessen
bewegt hat. Ohne diesen Nachweis wäre jede folgende Prüfung wertlos — man würde
die Qualität einer womöglich unvollständigen Ladung messen.

In [ ]:
manifeste = []
for teil in teile:
    m = json.loads((teil.parent / 'manifest.json').read_text(encoding='utf-8'))
    manifeste.append({
        'run_id': m['run_id'],
        'modus': m['mode'],
        'zeilen': m['row_count'],
        'verified': m['completeness']['verified'],
        'quelle_bewegt': m['completeness']['source_changed_during_run'],
        'erwartet_vorher': m['completeness']['source_count_before'],
        'geladen': m['completeness']['downloaded_rows'],
        'watermark': m['watermark']['maximum_seen'],
    })

laeufe = pd.DataFrame(manifeste).sort_values('run_id')
display(laeufe)

alle_verifiziert = bool(laeufe['verified'].all())
stimmig = bool((laeufe['erwartet_vorher'] == laeufe['geladen']).all())
print()
print('Alle Läufe verifiziert          :', alle_verifiziert)
print('Erwartete = geladene Zeilenzahl :', stimmig)

BEFUNDE['Ladevorgang nachgewiesen'] = (
    'ok' if alle_verifiziert and stimmig else 'PRUEFEN',
    f'{len(laeufe)} Läufe, alle mit id-Abgleich',
)

## Schritt 3 – Duplikate: der Delta-Überlapp

Die Ingestion lädt Delta-Läufe mit **14 Tagen Rückgriff**. Sie holt damit
absichtlich Datensätze erneut, die schon im vorherigen Lauf standen — lieber
doppelt als eine Lücke.

Für die Bronze-Schicht ist das korrekt. Für jede Auswertung heisst es: **zuerst
deduplizieren**, sonst zählt man Fälle mehrfach. Die Regel lautet, je `id` die
Version mit dem höchsten `updated_on` zu behalten.

In [ ]:
nach_id = defaultdict(list)
for zeile in roh:
    nach_id[zeile['id']].append(zeile)

mehrfach = {k: v for k, v in nach_id.items() if len(v) > 1}
echte_versionen = {
    k: v for k, v in mehrfach.items() if len({z['updated_on'] for z in v}) > 1
}

print(f'Zeilen roh              : {len(roh):,}')
print(f'eindeutige ids          : {len(nach_id):,}')
print(f'Zeilen zu viel          : {len(roh) - len(nach_id):,}')
print(f'ids mehrfach vorhanden  : {len(mehrfach):,}')
print(f'davon echte Versionen   : {len(echte_versionen):,}  (unterschiedliches updated_on)')
print()

# Deduplizieren: je id die zuletzt aktualisierte Version behalten.
daten = [
    max(versionen, key=lambda z: z['updated_on'] or '')
    for versionen in nach_id.values()
]
print(f'Datensätze nach Dedup   : {len(daten):,}')

if mehrfach:
    beispiel = next(iter(mehrfach.values()))
    print()
    print(f"Beispiel id={beispiel[0]['id']} — {len(beispiel)} Kopien:")
    for v in beispiel:
        print(f"   updated_on={v['updated_on']}  {v['primary_type']}")

BEFUNDE['Duplikate aus Delta-Überlapp'] = (
    'erwartet',
    f'{len(roh) - len(nach_id):,} Zeilen doppelt, nach Dedup {len(daten):,}',
)

## Schritt 4 – Welcher Schlüssel trägt

`id` und `case_number` sehen beide nach einem Schlüssel aus. Nur einer ist
einer.

`case_number` ist die Fallnummer der Polizei (RD-Nummer). Ein Fall kann
**mehrere** Straftaten umfassen — ein Doppelmord ist ein Fall mit zwei Opfern
und erscheint als zwei Zeilen. Wer über `case_number` verknüpft oder zählt,
verliert diese Zeilen oder vervielfacht sie.

In [ ]:
ids = [z['id'] for z in daten]
faelle = Counter(z['case_number'] for z in daten if z['case_number'])
mehrdeutig = {k: v for k, v in faelle.items() if v > 1}

print('id eindeutig nach Dedup :', len(ids) == len(set(ids)))
print('leere id                :', sum(1 for v in ids if not v))
print()
print(f'case_number mehrfach    : {len(mehrdeutig):,} Fallnummern')

if mehrdeutig:
    schluessel = next(iter(mehrdeutig))
    print()
    print(f'Beispiel {schluessel}:')
    for z in [z for z in daten if z['case_number'] == schluessel]:
        print(f"   id={z['id']:<10} {z['primary_type']} / {z['description']}")
    print()
    print('-> Ein Fall, mehrere Straftaten. id ist der Schlüssel, case_number nicht.')

BEFUNDE['Schlüssel id eindeutig'] = (
    'ok' if len(ids) == len(set(ids)) else 'FEHLER',
    f'{len(mehrdeutig)} case_number mehrfach — fachlich korrekt',
)

## Schritt 5 – Fehlende Werte je Spalte

Welche Spalten sind wie gut befüllt? Zu unterscheiden ist dabei zwischen
Spalten, die immer da sein müssen (`id`, `date`, `iucr`), und solchen, bei denen
ein Loch fachlich erklärbar ist (`location_description`).

In [ ]:
zeilen = []
for feld in dc.SCHEMA:
    fehlend = sum(1 for z in daten if z[feld.name] in (None, ''))
    zeilen.append({
        'Spalte': feld.name,
        'Typ': str(feld.type)[:28],
        'fehlend': fehlend,
        'Anteil %': round(fehlend / len(daten) * 100, 2),
    })

luecken = pd.DataFrame(zeilen).sort_values('fehlend', ascending=False)
display(luecken[luecken['fehlend'] > 0])

pflicht = ['id', 'case_number', 'date', 'iucr', 'primary_type', 'updated_on']
verletzt = luecken[luecken['Spalte'].isin(pflicht) & (luecken['fehlend'] > 0)]
print('Pflichtspalten ohne Lücke:', verletzt.empty)

BEFUNDE['Pflichtspalten befüllt'] = (
    'ok' if verletzt.empty else 'FEHLER',
    'id, date, iucr, primary_type, updated_on vollständig',
)

## Schritt 6 – Die Geo-Lücke, und warum sie nicht zufällig ist

Der wichtigste Befund des ganzen Notebooks.

Rund 0,81 % der Fälle haben **keine Koordinaten**. Das klingt nach einer
vernachlässigbaren Grösse — bis man fragt, *welche* Fälle betroffen sind. Die
Stadt unterdrückt die Verortung bei sensiblen Delikten, um Betroffene zu
schützen. Der Ausfall ist damit **systematisch nach Delikttyp verzerrt**.

**Die Quelle sagt es sogar selbst.** 334 Zeilen tragen den Distrikt `061`, den
Beat `6100` und die Community Area `0` — drei Kennungen, die es in Chicago
nicht gibt. **331 davon haben keine Koordinate**, und ihre häufigsten
Delikttypen sind *Criminal Sexual Assault*, *Offense Involving Children* und
*Sex Offense*. Das ist kein Erfassungsfehler, sondern ein Platzhalter: Wo der
Ort zurückgehalten wird, setzt Chicago auch die Gebietskennungen auf einen
Wert ausserhalb des Schemas. Diese 331 sind 16 % aller Zeilen ohne Koordinate;
die übrigen tragen ihren echten Distrikt.

Konsequenz für die geplante Tract-Auswertung: Fälle ohne Koordinaten lassen sich
keinem Census Tract zuordnen und fallen aus jeder räumlichen Auswertung heraus —
und zwar überproportional in genau den Kategorien, die häufig im Zentrum des
Interesses stehen. Das muss in jeder Ergebnisdarstellung stehen.


In [ ]:
ohne_geo = [z for z in daten if not z['latitude'] or not z['longitude']]
gesamt = Counter(z['primary_type'] for z in daten)
fehlend = Counter(z['primary_type'] for z in ohne_geo)

print(f'Ohne Koordinaten: {len(ohne_geo):,} von {len(daten):,} '
      f'({len(ohne_geo) / len(daten) * 100:.2f} %)')
print()

verteilung = pd.DataFrame([
    {
        'Delikttyp': typ,
        'ohne Geo': n,
        'gesamt': gesamt[typ],
        'Anteil %': round(n / gesamt[typ] * 100, 1),
    }
    for typ, n in fehlend.most_common()
]).sort_values('Anteil %', ascending=False)

display(verteilung.head(12))

schnitt = len(ohne_geo) / len(daten) * 100
auffaellig = verteilung[verteilung['Anteil %'] > schnitt * 3]
print(f'Durchschnittlicher Ausfall: {schnitt:.2f} %')
print(f'Delikttypen mit mehr als dem Dreifachen: {len(auffaellig)}')

BEFUNDE['Koordinaten vorhanden'] = (
    'ACHTUNG',
    f'{len(ohne_geo):,} Fälle ohne Geo ({schnitt:.2f} %), verzerrt nach Delikttyp',
)

## Schritt 7 – Wertebereiche und Gültigkeit

Stehen in den Spalten Werte, die überhaupt sein können? Geprüft werden

- `date` und `year`: liegt alles im Jahrgang, und stimmen beide überein?
- `updated_on`: nie **vor** dem Tatzeitpunkt
- Koordinaten: innerhalb der Stadtgrenzen, keine Nullinsel `(0, 0)`
- `community_area`: Chicago hat genau 77

In [ ]:
# Datum und Jahr
jahre_date = Counter(z['date'][:4] for z in daten if z['date'])
jahre_spalte = Counter(z['year'] for z in daten if z['year'])
uneinig = sum(1 for z in daten if z['date'] and z['year'] and z['date'][:4] != z['year'])
rueckdatiert = sum(
    1 for z in daten if z['updated_on'] and z['date'] and z['updated_on'] < z['date']
)

print('Jahr laut date-Spalte  :', dict(jahre_date))
print('Jahr laut year-Spalte  :', dict(jahre_spalte))
print('date und year uneinig  :', uneinig)
print('updated_on vor date    :', rueckdatiert)
print()

# Koordinaten
lat = [float(z['latitude']) for z in daten if z['latitude']]
lon = [float(z['longitude']) for z in daten if z['longitude']]
CHICAGO = {'lat': (41.60, 42.10), 'lon': (-87.95, -87.50)}
ausserhalb = sum(
    1 for a, b in zip(lat, lon)
    if not (CHICAGO['lat'][0] <= a <= CHICAGO['lat'][1]
            and CHICAGO['lon'][0] <= b <= CHICAGO['lon'][1])
)
nullinsel = sum(1 for a, b in zip(lat, lon) if a == 0 and b == 0)

print(f'latitude  {min(lat):.5f} bis {max(lat):.5f}')
print(f'longitude {min(lon):.5f} bis {max(lon):.5f}')
print('ausserhalb der Stadtgrenzen:', ausserhalb)
print('Nullinsel (0, 0)           :', nullinsel)
print()

# Community Areas
flaechen = {int(z['community_area']) for z in daten if z['community_area']}
print('community_area: von', min(flaechen), 'bis', max(flaechen),
      '| ausserhalb 1-77:', sum(1 for v in flaechen if not 1 <= v <= 77))

sauber = (uneinig == 0 and rueckdatiert == 0 and ausserhalb == 0 and nullinsel == 0)
BEFUNDE['Wertebereiche gültig'] = (
    'ok' if sauber else 'PRUEFEN',
    'Jahrgang, Zeitlogik, Koordinaten und community_area plausibel',
)

## Schritt 8 – Konsistenz zwischen Feldern

Einzelne Werte können gültig sein und trotzdem nicht zueinander passen. Zwei
Beziehungen lassen sich hart prüfen:

- **`beat` und `district`**: Ein Beat gehört zu einem District, seine ersten
  Stellen sind dessen Nummer. Beat `1424` gehört also zu District `014`.
- **`location` gegen `latitude`/`longitude`**: Dieselbe Angabe, zweimal
  geliefert — sie muss übereinstimmen.

Nebenbei fällt die Verteilung über die Distrikte auf: Chicago führt die
Streifenbezirke 001 bis 025, wobei 013, 021 und 023 aufgelöst wurden und
erwartungsgemäss fehlen. Darüber hinaus stehen **zwei** Nummern in den Daten,
und sie haben verschiedene Ursachen:

| | Fälle | was es ist |
| --- | ---: | --- |
| `031` | 14 | ein Erfassungsartefakt — die Beats darin gehören zu 016, 024 und 061 |
| `061` | 331 | **kein Artefakt**, sondern der Platzhalter für unterdrückte Orte (Schritt 6) |

Die Zelle prüft deshalb, ob die Widersprüche und der Kleinstdistrikt dieselbe
Ursache haben.


In [ ]:
abweichung = [
    z for z in daten
    if z['beat'] and z['district']
    and z['beat'][:-2].lstrip('0') != z['district'].lstrip('0')
]
print(f'beat-Präfix passt nicht zu district: {len(abweichung):,} von {len(daten):,}')
print()

bezirke = Counter(z['district'] for z in daten if z['district'])
print('Distrikte und ihre Fallzahlen:')
for bezirk, anzahl in sorted(bezirke.items()):
    hinweis = '  <- auffällig klein' if anzahl < 100 else ''
    print(f'   {bezirk}  {anzahl:>7,}{hinweis}')
print()
print('Hinweis: 013, 021 und 023 fehlen, weil diese Distrikte aufgelöst wurden.')
print()

# Hängen die beiden Auffälligkeiten zusammen?
klein = {b for b, n in bezirke.items() if n < 100}
aus_klein = [z for z in abweichung if z['district'] in klein]
rest = [z for z in abweichung if z['district'] not in klein]
print(f'Abweichungen aus den auffällig kleinen Distrikten {sorted(klein)}: {len(aus_klein)}')
for bezirk in sorted(klein):
    beats = Counter(z['beat'] for z in daten if z['district'] == bezirk)
    zeigt_auf = sorted({b[:-2].lstrip('0').zfill(3) for b in beats})
    print(f'   district={bezirk}: beats {dict(beats)} -> gehören zu {zeigt_auf}')
print()
print('übrige Abweichungen:', [(z['beat'], z['district']) for z in rest])
print()
print('-> Der Kleinstdistrikt ist ein Erfassungsartefakt. Wo beide Felder sich')
print('   widersprechen, ist beat das belastbarere.')
print()

struct_ab = sum(
    1 for z in daten
    if z['location'] and (
        (z['location'].get('latitude') or None) != (z['latitude'] or None)
        or (z['location'].get('longitude') or None) != (z['longitude'] or None)
    )
)
print('location-Struct weicht von latitude/longitude ab:', struct_ab)

BEFUNDE['Feldkonsistenz'] = (
    'ok' if struct_ab == 0 and len(abweichung) < len(daten) * 0.001 else 'PRUEFEN',
    f'{len(abweichung)} beat/district-Abweichungen, {struct_ab} Geo-Widersprüche',
)

## Schritt 9 – Fachliche Codes gegen die Referenz

`iucr` ist der Deliktschlüssel Chicagos und die Grundlage der NIBRS-Zuordnung.
Steht in den Daten ein Code, den `config/iucr_nibrs_mapping.csv` nicht kennt,
fällt der Fall bei der Harmonisierung mit Detroit stillschweigend heraus.

Die Codes sind vierstellig und **führende Nullen sind bedeutungstragend**:
`0110` ist Mord. Wer sie als Zahl einliest, zerstört den Schlüssel — ein Grund
mehr, warum Bronze alles als Text hält.

In [ ]:
mapping_datei = PROJECT_ROOT / 'config' / 'iucr_nibrs_mapping.csv'
with mapping_datei.open(encoding='utf-8-sig') as f:
    referenz = list(csv.DictReader(f))

spalte = next(c for c in referenz[0] if c.lower().strip() == 'iucr')
bekannt = {r[spalte].strip().upper().zfill(4) for r in referenz if r.get(spalte)}
print('Referenz:', mapping_datei.name, '|', len(bekannt), 'bekannte IUCR-Codes')

unbekannt = Counter(
    z['iucr'] for z in daten
    if z['iucr'] and z['iucr'].strip().upper().zfill(4) not in bekannt
)
print(f'Fälle mit unbekanntem Code: {sum(unbekannt.values()):,}')
if unbekannt:
    print('betroffene Codes:', unbekannt.most_common(10))
print()

paare = defaultdict(set)
for z in daten:
    if z['iucr'] and z['primary_type']:
        paare[z['iucr']].add(z['primary_type'])
uneindeutig = {k: v for k, v in paare.items() if len(v) > 1}
print('IUCR-Codes mit mehreren primary_type:', len(uneindeutig))
if uneindeutig:
    print('  ', list(uneindeutig.items())[:3])

BEFUNDE['IUCR gegen Referenz'] = (
    'ok' if not unbekannt else 'ACHTUNG',
    f'{sum(unbekannt.values())} Fälle ohne Zuordnung in der Mapping-Datei',
)

## Schritt 10 – Zeitliche Abdeckung und Aktualität

Zwei verschiedene Zeitachsen: `date` ist der Tatzeitpunkt, `updated_on` der
letzte Änderungszeitpunkt in der Quelle. Beide sagen etwas anderes über
Qualität aus.

Lücken in der monatlichen Verteilung wären ein Alarmzeichen — sie hiessen, dass
ein Zeitraum fehlt. Ein sehr altes `updated_on`-Maximum hiesse, dass der letzte
Lauf lange her ist.

In [ ]:
monate = Counter(z['date'][:7] for z in daten if z['date'])
verlauf = pd.DataFrame(
    sorted(monate.items()), columns=['Monat', 'Fälle']
).set_index('Monat')
display(verlauf.T)

erwartet = {f'{JAHR}-{m:02d}' for m in range(1, 13)}
fehlende = sorted(erwartet - set(monate))
print('Fehlende Monate:', fehlende or 'keine')

schnitt = sum(monate.values()) / len(monate)
schwach = {m: n for m, n in monate.items() if n < schnitt * 0.5}
print('Monate mit weniger als der Hälfte des Durchschnitts:', schwach or 'keine')
print()
print('jüngstes updated_on:', max(z['updated_on'] for z in daten if z['updated_on']))
print('ältestes updated_on:', min(z['updated_on'] for z in daten if z['updated_on']))

BEFUNDE['Zeitliche Abdeckung'] = (
    'ok' if not fehlende and not schwach else 'PRUEFEN',
    f'{len(monate)} Monate belegt, keine Lücke',
)

## Schritt 11 – Schema-Drift: was in `_extra` gelandet ist

`_extra` ist das Frühwarnsystem der Ingestion. Dort landet alles, was nicht ins
gepinnte Schema passt: ein Feld, das die Stadt neu ergänzt hat, oder ein Wert,
dessen JSON-Typ sich geändert hat.

Eine leere Spalte heisst: Die Quelle verhält sich noch genau so, wie das Schema
es beschreibt. Ist sie befüllt, sollte man hinsehen, **bevor** die Auswertung
darauf aufsetzt.

In [ ]:
mit_extra = [z for z in daten if z['_extra']]
print(f'Zeilen mit _extra: {len(mit_extra):,} von {len(daten):,}')

if mit_extra:
    felder = Counter()
    for z in mit_extra:
        felder.update(json.loads(z['_extra']).keys())
    print('Betroffene Felder:', felder.most_common())
    print('Beispiel:', mit_extra[0]['_extra'][:200])
    print()
    print('-> Schema-Drift. Vor der Auswertung klären, ob das Feld gebraucht wird.')
else:
    print('-> Kein Schema-Drift: die Quelle liefert genau die 22 bekannten Felder.')

# Der Fingerabdruck der Quelle im Manifest zeigt Typänderungen zusätzlich an.
fingerabdruecke = {
    json.loads((t.parent / 'manifest.json').read_text(encoding='utf-8'))
    ['source_schema']['fingerprint']
    for t in teile
}
print()
print('Schema-Fingerabdrücke über alle Läufe:', len(fingerabdruecke))
print('-> ', 'unverändert' if len(fingerabdruecke) == 1 else 'ACHTUNG: Quelle hat sich geändert')

BEFUNDE['Schema stabil'] = (
    'ok' if not mit_extra and len(fingerabdruecke) == 1 else 'ACHTUNG',
    f'_extra leer, {len(fingerabdruecke)} Fingerabdruck über alle Läufe',
)

## Schritt 12 – Die Befunde zusammen

Alle Prüfungen in einer Übersicht. `ok` heisst bestanden, `erwartet` ein
bekanntes und gewolltes Verhalten, `ACHTUNG` einen Befund, der die Auswertung
beeinflusst.

In [ ]:
bericht = pd.DataFrame(
    [{'Prüfung': k, 'Status': s, 'Befund': d} for k, (s, d) in BEFUNDE.items()]
)
display(bericht)

print('Datensätze nach Deduplizierung :', f'{len(daten):,}')
print('davon räumlich auswertbar      :',
      f'{len(daten) - len(ohne_geo):,}',
      f'({(len(daten) - len(ohne_geo)) / len(daten) * 100:.2f} %)')

## Ergebnis und was daraus für Silver folgt

Der Jahrgang 2024 ist in gutem Zustand: vollständig geladen und nachgewiesen,
keine Lücke in der Zeitachse, kein unbekannter Deliktschlüssel, kein
Schema-Drift, `id` als tragfähiger Schlüssel.

Vier Punkte muss die Silver-Schicht aber ausdrücklich behandeln:

| Befund | Was in Silver zu tun ist |
| --- | --- |
| Delta-Läufe überlappen bewusst | Nach `id` deduplizieren, je Fall die Version mit dem höchsten `updated_on` |
| 0,81 % ohne Koordinaten, **verzerrt nach Delikttyp** | Fälle ohne Geo getrennt ausweisen, nicht stillschweigend fallen lassen. Bei jeder Tract-Auswertung die Abdeckung je Delikttyp mitangeben |
| `case_number` ist kein Schlüssel | Immer über `id` verknüpfen und zählen |
| Alles ist Text | Zeitstempel parsen, `latitude`/`longitude` casten — und dabei die führenden Nullen von `iucr`, `beat`, `district` erhalten |

Dazu die kleineren Beobachtungen: **20 Zeilen**, bei denen `beat` und `district`
nicht zusammenpassen, und ein District `031` mit 14 Fällen, den es als
Streifenbezirk nicht gibt. Beides betrifft zu wenige Fälle, um Ergebnisse zu
verschieben, sollte aber bekannt sein, bevor jemand nach District gruppiert.

**Der District `061` gehört ausdrücklich nicht in diese Kategorie.** Seine 331
Fälle sind kein Artefakt, sondern Chicagos Platzhalter für zurückgehaltene Orte
— zusammen mit Beat `6100` und Community Area `0`. Wer nach District gruppiert
und ihn für einen Erfassungsfehler hält, wirft genau die Delikte weg, bei denen
die Stadt den Ort bewusst schützt.

**Nächster Schritt:** der räumliche Join gegen die TIGER-Tracts aus
`census_pipeline`. Dieses Notebook liefert dafür die Erwartung — wie viele Fälle
überhaupt zuordenbar sein können, und wo die Zuordnung systematisch fehlen wird.